<a href="https://colab.research.google.com/github/n2pakbong/boundary-adapted-pinns/blob/main/experiments/first_passage/duffing_oscillator/duffing_mean_escape_time.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Duffing oscillator mean escape time

This notebook solves the mean-exit-time equation for the stochastic Duffing oscillator in a disk. It trains a boundary-adapted PINN and evaluates the solution against Monte Carlo estimates. The example uses $\zeta=0.08$, $\epsilon=10^{-5}$, and $R=2$.


## 1. Problem

The stochastic Duffing oscillator is

$$
dX_1=X_2\,dt+\sqrt{2\epsilon}\,dB_1,
\qquad
dX_2=(-X_1-X_1^3-2\zeta X_2)\,dt+\sqrt{2\zeta}\,dB_2.
$$

For the first exit from the disk $\Omega=B_R(0)$, the mean escape time $\tau(x)=\mathbb E_x[T]$ solves

$$\mathcal L\tau=-1\quad\text{in }\Omega,\qquad \tau=0\quad\text{on }\partial\Omega,$$

$$\mathcal L\tau=x_2\tau_{x_1}-(x_1+x_1^3+2\zeta x_2)\tau_{x_2}
+\epsilon\tau_{x_1x_1}+\zeta\tau_{x_2x_2}.$$

No pretrained Duffing PINN result is bundled. Numerical values appear only after this notebook has been run.


In [ ]:
import json
import math
import random
import secrets
import time
from datetime import datetime, timezone
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    drive = None

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn

# Runtime settings are separate from the experiment parameters below.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE_NAME = "float64"  # Change to "float32" if needed.
if DTYPE_NAME not in ("float32", "float64"):
    raise ValueError("DTYPE_NAME must be 'float32' or 'float64'.")
torch.set_default_dtype(torch.float64 if DTYPE_NAME == "float64" else torch.float32)
print("Training device:", DEVICE)

## 2. Configure the experiment

Edit the run controls, physical parameters, total training points, Monte Carlo reference, optimizer, and evaluation settings in their labeled cells.

In [ ]:
# Run and output
RUN_MODE = "new"                 # "new" trains a fresh run; "resume" or "evaluate" selects a saved run.
RUN_LABEL = "duffing"
SAVE_TO_DRIVE = False            # True mounts Google Drive when the run folder is prepared; False uses ./runs/duffing.
RUN_LBFGS = False

### Duffing process and absorbing disk

The state is $X_t=(x_t,v_t)$. These values define the drift, diffusion, and disk in the exit-time PDE above.

In [ ]:
radius = 2.0
zeta = 0.08
epsilon = 1e-5
linear_stiffness = 1.0
cubic_stiffness = 1.0
boundary_power = 2               # Power in the trial function that enforces tau=0 at |x|=R.

## 3. Model and PDE

`BaseNetwork` below is ordinary PyTorch code. Edit its layers, activations, features, forward pass, or initialization directly. The wrapper multiplies its scalar output by the boundary-distance factor so the learned solution vanishes on the disk.

In [ ]:
def smooth_distance(points):
    return (radius**2 - points.square().sum(dim=1)) / (2 * radius)


class BaseNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(2, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 1),
        )
        self.layers.apply(self.initialize)

    @staticmethod
    def initialize(layer):
        if isinstance(layer, nn.Linear):
            nn.init.xavier_normal_(layer.weight)
            nn.init.zeros_(layer.bias)

    def forward(self, points):
        return self.layers(points)


class DuffingBoundaryPINN(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = BaseNetwork()

    def forward(self, points):
        raw = self.network(points)
        if raw.ndim == 1:
            raw = raw.unsqueeze(1)
        if raw.shape != (points.shape[0], 1):
            raise ValueError("BaseNetwork must return one scalar per point: [n] or [n, 1].")
        return raw.pow(boundary_power) * smooth_distance(points).clamp_min(0).unsqueeze(1)


### Duffing drift

Euler–Maruyama advances both components until the path exits the disk or reaches the time cap. This plot checks that the same equation and noise strengths used by the PDE are being simulated.


In [ ]:
def duffing_drift(points):
    x1, x2 = points[..., 0], points[..., 1]
    return torch.stack((x2, -linear_stiffness * x1 - cubic_stiffness * x1**3 - 2 * zeta * x2), dim=-1)


### PDE residual

The strong residual below is \(\mathcal L\tau_\theta+1\). Edit it together with the SDE coefficients and drift if you change the process.

In [ ]:
def residual(model, points):
    x = points.detach().clone().requires_grad_(True)
    tau = model(x).squeeze(1)
    grad = torch.autograd.grad(tau.sum(), x, create_graph=True)[0]
    tau_xx = torch.autograd.grad(grad[:, 0].sum(), x, create_graph=True)[0][:, 0]
    tau_yy = torch.autograd.grad(grad[:, 1].sum(), x, create_graph=True)[0][:, 1]
    operator = (
        x[:, 1] * grad[:, 0]
        - (linear_stiffness * x[:, 0] + cubic_stiffness * x[:, 0]**3 + 2 * zeta * x[:, 1]) * grad[:, 1]
        + epsilon * tau_xx + zeta * tau_yy
    )
    return operator + 1.0


def pde_loss(model, points):
    return residual(model, points).square().mean()


## 4. Sampling and reference data

`collocation_count` is the total number of interior points in the empirical PDE loss. The points are sampled once, saved, and reused at every optimizer update. The separate evaluation set is only for progress reports. No boundary points are needed because the trial function enforces zero on the disk.

In [ ]:
collocation_count = 512  # Total interior points used in the PDE loss each epoch.
evaluation_count = 512   # Separate fixed points used only for progress reporting.
# No boundary points are needed: the trial function imposes tau=0 on the circle exactly.

### Disk sampler

For a uniform sample by area in the disk, draw an angle uniformly and a radius \(R\sqrt U\), with \(U\sim\mathrm{Uniform}(0,1)\). This sampler supplies the fixed PDE and evaluation sets.

In [ ]:
def sample_disk(count, *, device=DEVICE):
    """Sample uniformly by area: angle is uniform and radius is R * sqrt(U)."""
    angle = 2 * math.pi * torch.rand(count, device=device)
    radial = radius * torch.sqrt(torch.rand(count, device=device))
    return torch.stack((radial * torch.cos(angle), radial * torch.sin(angle)), dim=1)


### Held-out Monte Carlo settings

Edit these values for this part of the experiment.

In [ ]:
mc_anchors = 20                      # Starting points where exit-time references are estimated.
mc_paths = 32                        # Independent paths from each label point.
mc_dt = 2e-3
mc_max_time = 5.0

### Optional trajectory settings

This simulation is for an optional illustration. Its parameters do not change the Monte Carlo labels or training points.

In [ ]:
trajectory_start = (0.0, 0.0)
trajectory_dt = 1e-3
trajectory_max_time = 5.0
trajectory_seed_offset = 1

## 5. Training objectives and controls

Edit these values for this part of the experiment.

In [ ]:
learning_rate = 1e-2
adam_beta_1 = 0.9
adam_beta_2 = 0.999
adam_epsilon = 1e-8
weight_decay = 0.0
adam_epochs = 40
final_learning_rate_factor = 5e-2
report_every = 10
checkpoint_every = 10

### Optimizer and learning-rate schedule

Edit these values for this part of the experiment.

In [ ]:
lbfgs_iterations = 20
lbfgs_lr = 1.0
lbfgs_history_size = 100
lbfgs_tolerance_grad = 0.0
lbfgs_tolerance_change = 0.0
evaluation_grid_n = 180
plot_dpi = 180

## 6. Prepare and train

A new run gets a unique folder and fresh seeds. For resume or evaluation, the notebook finds saved runs automatically; if there is more than one, it lists them and asks for a run number.

In [ ]:
if RUN_MODE not in {"new", "resume", "evaluate"}:
    raise ValueError("RUN_MODE must be 'new', 'resume', or 'evaluate'.")

if SAVE_TO_DRIVE:
    if drive is None:
        raise RuntimeError("Google Drive saving is available in Colab; set SAVE_TO_DRIVE = False for local runs.")
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive", force_remount=False)
    output_base = Path("/content/drive/MyDrive/boundary-adapted-pinns-runs/duffing")
else:
    output_base = Path("runs/duffing")
output_base.mkdir(parents=True, exist_ok=True)

if RUN_MODE == "new":
    # These seeds are generated once for this new run and saved in its manifest.
    SEED = secrets.randbits(32)
    MC_SEED = secrets.randbits(32)
    run_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S.%fZ")
    run_dir = output_base / f"{RUN_LABEL}_{run_stamp}_seed{SEED}"
    run_dir.mkdir(parents=False, exist_ok=False)
    saved_manifest = None
else:
    candidates = sorted(
        (path for path in output_base.glob(f"{RUN_LABEL}_*") if path.is_dir() and (path / "run_manifest.json").is_file()),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    if not candidates:
        raise FileNotFoundError(f"No saved run for {RUN_LABEL!r} was found in {output_base}.")
    if len(candidates) == 1:
        run_dir = candidates[0]
    else:
        print("Choose the saved run:")
        for number, path in enumerate(candidates, start=1):
            print(f"  {number}. {path.name}")
        choice = int(input(f"Run number [1-{len(candidates)}]: "))
        if not 1 <= choice <= len(candidates):
            raise ValueError("Choose one of the listed run numbers.")
        run_dir = candidates[choice - 1]
    manifest_path = run_dir / "run_manifest.json"
    saved_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    SEED = int(saved_manifest["seed"])
    MC_SEED = int(saved_manifest.get("mc_seed", saved_manifest.get("monte_carlo_seed")))

RESUME = RUN_MODE == "resume"
LOAD_SAVED_MODELS = RUN_MODE == "evaluate"
print(f"Mode: {RUN_MODE} | output: {run_dir} | training seed: {SEED} | Monte Carlo seed: {MC_SEED}")

def seed_all(seed):
    random.seed(int(seed))
    np.random.seed(int(seed) % (2**32))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))

def atomic_torch_save(payload, path):
    temporary_path = path.with_name(path.name + ".tmp")
    torch.save(payload, temporary_path)
    temporary_path.replace(path)


def atomic_csv_save(frame, path):
    temporary_path = path.with_name(path.name + ".tmp")
    frame.to_csv(temporary_path, index=False)
    temporary_path.replace(path)


def capture_rng_state():
    return {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
        "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }

def restore_rng_state(state):
    if not state:
        raise ValueError("Checkpoint has no random-generator state; exact resume is unavailable.")
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"].cpu())
    if torch.cuda.is_available() and state.get("cuda") is not None:
        torch.cuda.set_rng_state_all([item.cpu() for item in state["cuda"]])

seed_all(SEED)


### Run record

The next cell writes `run_manifest.json` in the selected run folder. It records the run seeds, the values that affect the experiment, its status, and the names of saved files. For resume or evaluation, it checks that the selected folder was created with matching settings and stops if they differ. Change experiment values in the cells where those values are introduced; this cell only records and checks them.

In [ ]:
experiment_settings = {
    "run_label": RUN_LABEL, "seed": SEED, "mc_seed": MC_SEED,
    "radius": radius, "zeta": zeta, "epsilon": epsilon,
    "linear_stiffness": linear_stiffness, "cubic_stiffness": cubic_stiffness,
    "boundary_power": boundary_power,
    "collocation_count": collocation_count, "evaluation_count": evaluation_count,
    "mc_anchors": mc_anchors, "mc_paths": mc_paths, "mc_dt": mc_dt, "mc_max_time": mc_max_time,
    "learning_rate": learning_rate, "adam_beta_1": adam_beta_1, "adam_beta_2": adam_beta_2,
    "adam_epsilon": adam_epsilon, "weight_decay": weight_decay, "adam_epochs": adam_epochs,
    "final_learning_rate_factor": final_learning_rate_factor,
    "run_lbfgs": RUN_LBFGS, "lbfgs_iterations": lbfgs_iterations, "lbfgs_lr": lbfgs_lr,
    "lbfgs_history_size": lbfgs_history_size, "evaluation_grid_n": evaluation_grid_n,
    "device": str(DEVICE), "dtype": DTYPE_NAME, "plot_dpi": plot_dpi,
    "trajectory_start": trajectory_start, "trajectory_dt": trajectory_dt, "trajectory_max_time": trajectory_max_time,
}
experiment_settings = json.loads(json.dumps(experiment_settings))

def require_matching_settings(record, expected, saved_path):
    saved = record.get("settings", record)
    if saved != expected:
        raise ValueError(
            f"{saved_path} was created with different settings. "
            "Restore those settings or start a new run."
        )

manifest_path = run_dir / "run_manifest.json"
if RUN_MODE == "new":
    write_run_manifest_record = {
        "schema_version": 2,
        "run_id": run_dir.name,
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "seed": SEED,
        "mc_seed": MC_SEED,
        "monte_carlo_seed": MC_SEED,
        "settings": experiment_settings,
        "status": "created",
        "artifacts": [],
    }
    with manifest_path.open("x", encoding="utf-8") as stream:
        json.dump(write_run_manifest_record, stream, indent=2)
else:
    write_run_manifest_record = json.loads(manifest_path.read_text(encoding="utf-8"))
    require_matching_settings(write_run_manifest_record, experiment_settings, manifest_path)

def write_run_manifest(status):
    write_run_manifest_record["status"] = status
    write_run_manifest_record["updated_utc"] = datetime.now(timezone.utc).isoformat()
    write_run_manifest_record["artifacts"] = sorted(
        str(path.relative_to(run_dir))
        for path in run_dir.rglob("*")
        if path.is_file() and path != manifest_path and ".tmp" not in path.name
    )
    temporary_path = manifest_path.with_name(manifest_path.name + ".tmp")
    temporary_path.write_text(json.dumps(write_run_manifest_record, indent=2), encoding="utf-8")
    temporary_path.replace(manifest_path)

def atomic_npz_save(path, **arrays):
    temporary_path = path.with_name(path.name + ".tmp")
    with temporary_path.open("wb") as stream:
        np.savez_compressed(stream, **arrays)
    temporary_path.replace(path)

def atomic_figure_save(figure, path, **kwargs):
    temporary_path = path.with_name(path.stem + ".tmp" + path.suffix)
    figure.savefig(temporary_path, **kwargs)
    temporary_path.replace(path)

def require_artifact_files(paths):
    missing = [path for path in paths if not path.is_file() or path.stat().st_size == 0]
    if missing:
        raise RuntimeError("Missing or empty run outputs: " + ", ".join(map(str, missing)))

write_run_manifest("preparing")
print(f"Run folder: {run_dir} | seed: {SEED} | Monte Carlo seed: {MC_SEED}")


### Train and save

A new run or resumed run trains on every point in the saved collocation set each epoch. The checkpoint includes the model, optimizer, schedule, history, and random-generator state. Evaluation writes the Monte Carlo reference, solution/error arrays, metrics, and figures to the selected run folder.

In [ ]:
training_points_path = run_dir / "training_points.pt"
if RUN_MODE in {"resume", "evaluate"}:
    if not training_points_path.is_file():
        raise FileNotFoundError(f"Saved training points are missing: {training_points_path}")
    point_record = torch.load(training_points_path, map_location=DEVICE, weights_only=False)
    require_matching_settings(point_record, experiment_settings, training_points_path)
    collocation_points = point_record["collocation"].to(DEVICE)
    evaluation_points = point_record["evaluation"].to(DEVICE)
    print("Loaded this run's training and evaluation points:", training_points_path)
else:
    if training_points_path.exists():
        raise FileExistsError(f"Refusing to overwrite existing run data: {training_points_path}")
    seed_all(MC_SEED)
    collocation_points = sample_disk(collocation_count)
    evaluation_points = sample_disk(evaluation_count)
    training_points = {
        "collocation": collocation_points.cpu(),
        "evaluation": evaluation_points.cpu(),
        "settings": experiment_settings,
    }
    atomic_torch_save(training_points, training_points_path)
    print("Saved this run's training and evaluation points:", training_points_path)

atomic_npz_save(
    run_dir / "training_points.npz",
    collocation=collocation_points.cpu().numpy(),
    evaluation=evaluation_points.cpu().numpy(),
    training_seed=np.asarray(SEED, dtype=np.uint64),
    monte_carlo_seed=np.asarray(MC_SEED, dtype=np.uint64),
)
seed_all(SEED)

last_path = run_dir / "last.pt"
best_path = run_dir / "best.pt"
final_path = run_dir / "final.pt"


def train_duffing():
    if RUN_MODE == "resume" and final_path.exists():
        raise ValueError(f"{final_path} is already complete; set RUN_MODE='evaluate' to reuse it.")
    model = DuffingBoundaryPINN().to(DEVICE)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        betas=(adam_beta_1, adam_beta_2),
        eps=adam_epsilon,
        weight_decay=weight_decay,
    )
    scheduler = torch.optim.lr_scheduler.ExponentialLR(
        optimizer,
        gamma=final_learning_rate_factor ** (1.0 / adam_epochs),
    )
    history, start_epoch, best_loss = [], 0, math.inf
    lbfgs_history = []
    if RUN_MODE == "resume" and last_path.exists():
        state = torch.load(last_path, map_location=DEVICE, weights_only=False)
        require_matching_settings(state, experiment_settings, last_path)
        model.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        scheduler.load_state_dict(state["scheduler"])
        history, start_epoch, best_loss = state["history"], state["epoch"], state["best_loss"]
        restore_rng_state(state.get("rng_state"))
        print(f"Resuming after epoch {start_epoch}")
    elif RUN_MODE == "resume" and best_path.exists():
        raise ValueError(f"{best_path} exists without a resumable last.pt checkpoint; refusing an inexact resume.")
    elif RUN_MODE == "new" and (last_path.exists() or best_path.exists() or final_path.exists()):
        raise FileExistsError(f"Refusing to reuse checkpoints in {run_dir}")

    started = time.time()
    for epoch in range(start_epoch + 1, adam_epochs + 1):
        optimizer.zero_grad()
        loss = pde_loss(model, collocation_points)
        loss.backward()
        optimizer.step()
        scheduler.step()
        validation = float(pde_loss(model, evaluation_points).detach())
        row = {"epoch": epoch, "training residual": float(loss.detach()), "evaluation residual": validation}
        history.append(row)
        if validation < best_loss:
            best_loss = validation
            best_checkpoint = {
                "model": model.state_dict(),
                "epoch": epoch,
                "loss": validation,
                "settings": experiment_settings,
            }
            atomic_torch_save(best_checkpoint, best_path)
        if epoch % report_every == 0 or epoch == 1 or epoch == adam_epochs:
            print(
                f"epoch {epoch:6d}/{adam_epochs} | "
                f"train {row['training residual']:.3e} | eval {validation:.3e} | "
                f"{time.time() - started:.1f}s",
                flush=True,
            )
        if epoch % checkpoint_every == 0 or epoch == adam_epochs:
            atomic_torch_save({
                "model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(), "history": history, "epoch": epoch,
                "best_loss": best_loss, "settings": experiment_settings,
                "rng_state": capture_rng_state(),
            }, last_path)
            atomic_csv_save(pd.DataFrame(history), run_dir / "training_history.csv")
            write_run_manifest("training")

    model.load_state_dict(torch.load(best_path, map_location=DEVICE, weights_only=False)["model"])
    if RUN_LBFGS:
        optimizer_lbfgs = torch.optim.LBFGS(
            model.parameters(),
            lr=lbfgs_lr,
            max_iter=lbfgs_iterations,
            history_size=lbfgs_history_size,
            tolerance_grad=lbfgs_tolerance_grad,
            tolerance_change=lbfgs_tolerance_change,
            line_search_fn="strong_wolfe",
        )
        calls = 0

        def closure():
            nonlocal calls
            calls += 1
            optimizer_lbfgs.zero_grad()
            loss = pde_loss(model, collocation_points)
            loss.backward()
            lbfgs_history.append({"call": calls, "residual": float(loss.detach())})
            if calls % 100 == 0:
                atomic_csv_save(pd.DataFrame(lbfgs_history), run_dir / "lbfgs_history.csv")
                write_run_manifest("training")
                print(f"L-BFGS call {calls}: residual {float(loss.detach()):.3e}", flush=True)
            return loss

        optimizer_lbfgs.step(closure)
        atomic_csv_save(pd.DataFrame(lbfgs_history), run_dir / "lbfgs_history.csv")
        write_run_manifest("training")
    atomic_torch_save({
        "model": model.state_dict(), "history": history, "lbfgs_history": lbfgs_history,
        "run_label": RUN_LABEL, "parameters": {"zeta": zeta, "epsilon": epsilon, "radius": radius},
        "settings": {**experiment_settings, "lbfgs": RUN_LBFGS, "lbfgs_iterations": lbfgs_iterations},
        "rng_state": capture_rng_state(),
    }, final_path)
    history_frame = pd.DataFrame(history)
    atomic_csv_save(history_frame, run_dir / "training_history.csv")
    return model, history_frame


model = history = None
if RUN_MODE in {"new", "resume"}:
    write_run_manifest("training")
    if RUN_MODE == "evaluate":
        raise ValueError("Set RUN_MODE to 'new' or 'resume' before training.")
    model, history = train_duffing()
    display(history.tail())
    write_run_manifest("trained")
elif RUN_MODE == "evaluate":
    write_run_manifest("evaluating")
    if not final_path.is_file():
        raise FileNotFoundError(f"Completed model is missing: {final_path}")
    state = torch.load(final_path, map_location=DEVICE, weights_only=False)
    expected = {**experiment_settings, "lbfgs": RUN_LBFGS, "lbfgs_iterations": lbfgs_iterations}
    require_matching_settings(state, expected, final_path)
    model = DuffingBoundaryPINN().to(DEVICE)
    model.load_state_dict(state["model"])
    history = pd.DataFrame(state["history"])
    lbfgs_history = pd.DataFrame(state.get("lbfgs_history", []))
    if RUN_LBFGS and not lbfgs_history.empty:
        atomic_csv_save(lbfgs_history, run_dir / "lbfgs_history.csv")
    print("Loaded completed model from this run:", final_path)


if history is not None:
    atomic_csv_save(history, run_dir / "training_history.csv")

## 7. Evaluate and report

### Optional sample trajectory

This single Euler–Maruyama path is a visual check of the SDE and exit rule. It is separate from both PDE training and the Monte Carlo evaluation.

In [ ]:
def simulate_trajectory(
    start=trajectory_start, *, zeta_value=zeta, epsilon_value=epsilon,
    radius_value=radius, dt=trajectory_dt, max_time=trajectory_max_time, seed=SEED + trajectory_seed_offset,
):
    rng = np.random.default_rng(seed)
    point = np.asarray(start, dtype=float).copy()
    path = [point.copy()]
    exited = np.linalg.norm(point) >= radius_value
    noise = np.array([math.sqrt(2 * epsilon_value * dt), math.sqrt(2 * zeta_value * dt)])
    for _ in range(int(max_time / dt)):
        if exited:
            break
        drift_second = (
            -linear_stiffness * point[0]
            - cubic_stiffness * point[0]**3
            - 2 * zeta_value * point[1]
        )
        drift = np.array([point[1], drift_second])
        point = point + drift * dt + noise * rng.standard_normal(2)
        path.append(point.copy())
        exited = np.linalg.norm(point) >= radius_value
    return np.asarray(path), (len(path) - 1) * dt, exited


trajectory, trajectory_time, trajectory_exited = simulate_trajectory()
angle = np.linspace(0, 2 * np.pi, 300)
plt.figure(figsize=(6, 6))
plt.plot(trajectory[:, 0], trajectory[:, 1], lw=1)
plt.plot(radius * np.cos(angle), radius * np.sin(angle), "k--")
plt.scatter(*trajectory[0], label="start")
plt.scatter(*trajectory[-1], marker="x", label="last point")
plt.gca().set_aspect("equal")
plt.legend()
plt.title(f"Duffing trajectory: {'exit' if trajectory_exited else 'censored'} at t={trajectory_time:.3g}")
plt.show()


### Monte Carlo evaluation

This notebook trains with the PDE residual only. These independent Euler–Maruyama paths are generated after training to evaluate the learned solution at sampled anchors; they are not training labels. The anchor count, paths per anchor, time step, and time cap are set above.


In [ ]:
@torch.no_grad()
def monte_carlo_exit_time(starts, n_paths, dt, max_time):
    starts = starts.to(DEVICE)
    count = starts.shape[0]
    positions = starts[:, None, :].expand(count, n_paths, 2).clone()
    elapsed = torch.zeros(count, n_paths, device=DEVICE)
    alive = torch.ones(count, n_paths, dtype=torch.bool, device=DEVICE)
    steps = int(max_time / dt)
    interval = max(1, steps // 100)
    noise_scale = torch.tensor([math.sqrt(2 * epsilon * dt), math.sqrt(2 * zeta * dt)], device=DEVICE)
    started = time.time()
    for step in range(steps):
        if not alive.any():
            break
        current = positions[alive]
        positions[alive] = current + duffing_drift(current) * dt + torch.randn_like(current) * noise_scale
        elapsed[alive] += dt
        active_indices = alive.nonzero(as_tuple=False)
        exited = positions[alive].square().sum(dim=1) >= radius**2
        if exited.any():
            ids = active_indices[exited]
            alive[ids[:, 0], ids[:, 1]] = False
        if step % interval == 0 or step + 1 == steps:
            print(
                f"MC step {step + 1:7d}/{steps} | "
                f"alive {alive.float().mean():6.2%} | {time.time() - started:6.1f}s",
                flush=True,
            )
    return elapsed.mean(dim=1), alive.float().mean(dim=1)


write_run_manifest("preparing")
mc_path = run_dir / "monte_carlo_reference.pt"
mc_settings = {
    "format_version": 2,
    "seed": SEED, "mc_seed": MC_SEED,
        "monte_carlo_seed": MC_SEED, "radius": radius, "zeta": zeta, "epsilon": epsilon,
    "anchors": mc_anchors, "paths": mc_paths, "dt": mc_dt, "max_time": mc_max_time,
}
if RUN_MODE in {"resume", "evaluate"}:
    if not mc_path.is_file():
        raise FileNotFoundError(f"Saved Monte Carlo data is missing: {mc_path}")
    mc_record = torch.load(mc_path, map_location=DEVICE, weights_only=False)
    require_matching_settings(mc_record, mc_settings, mc_path)
    mc_x = mc_record["x"].to(DEVICE)
    mc_tau = mc_record["tau"].to(DEVICE)
    mc_censored = mc_record["censored"].to(DEVICE)
    print("Loaded Monte Carlo reference:", mc_path)
else:
    if mc_path.exists():
        raise FileExistsError(f"Refusing to overwrite existing Monte Carlo data: {mc_path}")
    seed_all(MC_SEED)
    mc_x = sample_disk(mc_anchors)
    mc_tau, mc_censored = monte_carlo_exit_time(mc_x, mc_paths, mc_dt, mc_max_time)
    monte_carlo_record = {
        "x": mc_x.cpu(),
        "tau": mc_tau.cpu(),
        "censored": mc_censored.cpu(),
        "settings": mc_settings,
    }
    atomic_torch_save(monte_carlo_record, mc_path)
    print("Saved Monte Carlo reference:", mc_path)

mc_frame = pd.DataFrame({
    "x1": mc_x[:, 0].cpu().numpy(), "x2": mc_x[:, 1].cpu().numpy(),
    "mean exit time": mc_tau.cpu().numpy(),
    "censored fraction": mc_censored.cpu().numpy(),
})
atomic_csv_save(mc_frame, run_dir / "monte_carlo_reference.csv")
write_run_manifest("evaluating" if model is not None else "prepared")

display(mc_frame.describe())


### Learned solution and errors

The contour plot shows the learned field. At the Monte Carlo anchors, the scatter plot and error table compare like with like. Large censoring fractions indicate that `mc_max_time` or the path count should be increased before drawing a numerical conclusion.


In [ ]:
if model is not None:
    write_run_manifest("evaluating")
    grid_n = evaluation_grid_n
    gx, gy = np.meshgrid(np.linspace(-radius, radius, grid_n), np.linspace(-radius, radius, grid_n))
    mask = gx**2 + gy**2 <= radius**2
    points = np.column_stack((gx[mask], gy[mask]))
    with torch.no_grad():
        prediction = model(torch.as_tensor(points, device=DEVICE)).squeeze(1).cpu().numpy()
        anchor_prediction = model(mc_x).squeeze(1).cpu().numpy()
    field = np.full_like(gx, np.nan)
    field[mask] = prediction
    truth = mc_tau.cpu().numpy()
    comparison = pd.DataFrame({
        "PINN": anchor_prediction,
        "Monte Carlo": truth,
        "absolute error": np.abs(anchor_prediction - truth),
        "censored fraction": mc_censored.cpu().numpy(),
    })
    display(comparison)
    comparison.to_csv(run_dir / "anchor_comparison.csv", index=False)
    anchor_rmse = float(np.sqrt(np.mean((anchor_prediction - truth)**2)))
    print("Anchor RMSE:", anchor_rmse)
    signed_anchor_error = anchor_prediction - truth
    absolute_anchor_error = np.abs(signed_anchor_error)
    atomic_npz_save(
        run_dir / "evaluation_field.npz",
        gx=gx, gy=gy, points=points, mask=mask.astype(bool), field=field,
        anchors=mc_x.cpu().numpy(), prediction=anchor_prediction, reference=truth,
        censored_fraction=mc_censored.cpu().numpy(),
        signed_anchor_error=signed_anchor_error,
        absolute_anchor_error=absolute_anchor_error,
    )

    fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
    contour = axes[0].contourf(gx, gy, field, levels=50)
    axes[0].set_title("Boundary-adapted PINN")
    axes[0].set_aspect("equal")
    fig.colorbar(contour, ax=axes[0])
    axes[1].scatter(truth, anchor_prediction, c=mc_censored.cpu().numpy(), cmap="magma")
    lo, hi = min(truth.min(), anchor_prediction.min()), max(truth.max(), anchor_prediction.max())
    axes[1].plot([lo, hi], [lo, hi], "k--")
    axes[1].set(
        xlabel="Monte Carlo mean exit time",
        ylabel="PINN",
        title="Anchor comparison; color = censored fraction",
    )
    atomic_figure_save(fig, run_dir / "evaluation.png", dpi=plot_dpi, bbox_inches="tight")
    anchor_mae = float(np.mean(absolute_anchor_error))
    anchor_relative_l2 = float(np.linalg.norm(signed_anchor_error) / max(np.linalg.norm(truth), np.finfo(float).eps))
    metrics_record = {
        "run_id": run_dir.name, "training_seed": SEED, "monte_carlo_seed": MC_SEED,
        "settings": experiment_settings,
        "anchor_rmse": anchor_rmse, "anchor_mae": anchor_mae,
        "anchor_relative_l2": anchor_relative_l2,
        "censored_fraction_mean": float(mc_censored.float().mean()),
        "arrays_file": "evaluation_field.npz",
    }
    metrics_path = run_dir / "evaluation_metrics.json"
    temporary_metrics_path = metrics_path.with_name(metrics_path.name + ".tmp")
    temporary_metrics_path.write_text(json.dumps(metrics_record, indent=2), encoding="utf-8")
    temporary_metrics_path.replace(metrics_path)
    write_run_manifest("evaluating")
    required_paths = [
        run_dir / "run_manifest.json",
        run_dir / "training_points.pt", run_dir / "training_points.npz",
        run_dir / "monte_carlo_reference.pt", run_dir / "monte_carlo_reference.csv",
        run_dir / "best.pt", run_dir / "last.pt", run_dir / "final.pt",
        run_dir / "training_history.csv",
        run_dir / "anchor_comparison.csv",
        run_dir / "evaluation_field.npz", run_dir / "evaluation.png", metrics_path,
    ]
    if RUN_LBFGS:
        required_paths.append(run_dir / "lbfgs_history.csv")
    require_artifact_files(required_paths)
    required_keys = {
        "gx", "gy", "points", "mask", "field", "anchors", "prediction",
        "reference", "censored_fraction", "signed_anchor_error", "absolute_anchor_error",
    }
    with np.load(run_dir / "training_points.npz", allow_pickle=False) as saved_points:
        required_point_arrays = {
            "collocation", "evaluation", "training_seed", "monte_carlo_seed"
        }
        missing_points = required_point_arrays.difference(saved_points.files)
    if missing_points:
        missing_list = ", ".join(sorted(missing_points))
        raise RuntimeError(
            "Training-point archive is incomplete; missing arrays: " + missing_list
        )
    with np.load(run_dir / "evaluation_field.npz", allow_pickle=False) as saved_arrays:
        missing_keys = required_keys.difference(saved_arrays.files)
    if missing_keys:
        raise RuntimeError("Evaluation archive is incomplete; missing arrays: " + ", ".join(sorted(missing_keys)))
    write_run_manifest("complete")
    plt.show()


## 8. Reuse and adapt

Replace the drift, diffusion coefficients, domain sampler, distance factor, and strong residual together; derive the new generator first so the residual signs and second-order coefficients stay consistent. The trajectory and Monte Carlo functions should use the same SDE and exit rule as the PDE. `make_base_network()` accepts any scalar PyTorch model, while `DuffingBoundaryPINN` keeps the nonnegative exact-boundary trial function separate from that architectural choice.


## References

- J. Qian, L. Chen, and J. Q. Sun, [“A neural network solution of first-passage problems”](https://doi.org/10.1007/s10483-024-3189-8), *Applied Mathematics and Mechanics (English Edition)* 45(11), 2023–2036 (2024).
- N. Tepakbong, J. Fan, X. Zhou, and D.-X. Zhou, [“Boundary-Adapted PINNs for Elliptic Dirichlet Problems”](https://arxiv.org/abs/2607.19167), 2026.
- N. Sukumar and A. Srivastava, [“Exact imposition of boundary conditions with distance functions”](https://doi.org/10.1016/j.cma.2021.114333), *CMAME* 389 (2022).
- M. Raissi, P. Perdikaris, and G. E. Karniadakis, [“Physics-informed neural networks”](https://doi.org/10.1016/j.jcp.2018.10.045), *JCP* 378 (2019).
